In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('data/insurance_data.csv', sep='|', low_memory=False)
df.columns = df.columns.str.strip()

# ====================== DATA PREPARATION ======================
# Create target and features
df_model = df[df['TotalClaims'] > 0].copy()  # Only policies with claims for severity model

# Feature Engineering
df_model['VehicleAge'] = 2015 - df_model['RegistrationYear']  # assuming data is up to 2015
df_model['PremiumPerSumInsured'] = df_model['TotalPremium'] / df_model['SumInsured'].replace(0, np.nan)

# Select features
features = ['VehicleAge', 'SumInsured', 'CalculatedPremiumPerTerm', 'Province', 
            'Gender', 'VehicleType', 'make', 'CoverType']

X = df_model[features].copy()
y = df_model['TotalClaims']

# Encode categorical variables
X = pd.get_dummies(X, drop_first=True)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# ====================== MODEL TRAINING ======================
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append({'Model': name, 'RMSE': round(rmse, 2), 'R²': round(r2, 4)})

# Results Table
results_df = pd.DataFrame(results)
display(results_df.sort_values('R²', ascending=False))

# ====================== BEST MODEL FEATURE IMPORTANCE ======================
best_model = models["Random Forest"]
best_model.fit(X_train, y_train)

importances = pd.Series(best_model.feature_importances_, index=X.columns)
importances = importances.nlargest(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Top 10 Most Important Features (Random Forest)')
plt.xlabel('Feature Importance')
plt.show()

print("\nTop 10 Important Features:")
print(importances)

ModuleNotFoundError: No module named 'xgboost'

In [2]:
import sys
print(sys.executable)

c:\Users\PC\AppData\Local\Programs\Python\Python312\python.exe


In [20]:
import xgboost
print(xgboost.__version__)

ModuleNotFoundError: No module named 'xgboost'